---
## **Exercise: Recommendation System (Content Based Filtering)**

**Gunakan dataset anime.csv**

**1. Cosine Similarity**
- Seorang user telah menonton 'Kimi No Nawa'. Berikan rekomendasi anime menggunakan Cosine Similarity!


**2. Content based filtering for one user**
- Pilih 3 anime yang disukai user (bebas), contoh: 'Naruto', 'One Piece', 'Dragon Ball'
- Buat user feature vector
- Cari 10 rekomendasi anime untuk user


**3. Rekomendasikan masing-masing 10 anime untuk setiap user**

Buatlah:
- Item feature: anime_id, name, genre
- Buat user-item rating matrix di mana isinya adalah 4 user dan 4 anime yang sudah diberi rating oleh keempat user tersebut (buat secara random saja)
        
        Contoh:
        
        df_user_items = pd.DataFrame({
        'user': ['user 1', 'user 2', 'user 3', 'user 4'],
        951:     [9, 4, 6, 7],
        136:    [9, 7,  9, 7],
        235:    [7, 8,  9, 7],
        918:    [5, 9,  5, 7]
        })

            
- Buat item-feature matrix untuk semua anime dan pilih 4 anime yang sudah diberi rating di atas
- Buat user feature matrix dan user feature vector-nya
- Buat rekomendasi untuk anime yang belum ditonton
- Sort dan filtering 10 rekomendasi anime untuk tiap user

In [75]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 30)
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

In [32]:
df = pd.read_csv('movie_dataset.csv', index_col = 0)
display(df.sample(2))
display(df.info())

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
index,,,,,,,,,,,,,,,,,,,,,,,
3823,3600000,Drama,NaN,3116,shower,en,Midnight Cowboy,A naive male prostitute and his sickly friend ...,21.119209,"[{""name"": ""United Artists"", ""id"": 60}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",1969-05-25,44785053,113.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Whatever you hear about Midnight Cowboy is true.,Midnight Cowboy,7.4,299,Dustin Hoffman Jon Voight Sylvia Miles John Mc...,"[{'name': 'John Barry', 'gender': 0, 'departme...",John Schlesinger
2287,0,Drama Comedy,NaN,26367,aunt duringcreditsstinger,en,I Can Do Bad All By Myself,When Madea catches sixteen-year-old Jennifer a...,2.242809,"[{""name"": ""Lions Gate Films"", ""id"": 35}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-09-11,0,113.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,I Can Do Bad All By Myself,6.0,40,Tyler Perry Taraji P. Henson Adam Rodr\u00edgu...,"[{'name': 'Tyler Perry', 'gender': 2, 'departm...",Tyler Perry


<class 'pandas.core.frame.DataFrame'>
Index: 4803 entries, 0 to 4802
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4775 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4391 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status                4803

None

In [33]:
df.fillna('', inplace = True)

for cols in df.columns:
    val_count = df[cols].value_counts()
    
    if len(val_count) <= 20:
        print(val_count)

status
Released           4795
Rumored               5
Post Production       3
Name: count, dtype: int64


C:\Users\benjamin\AppData\Local\Temp\ipykernel_18172\3650518989.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna('', inplace = True)


In [38]:
df['combined_features'] = df['genres'].fillna('') + ' ' + df['title'].fillna('')

In [70]:
# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['combined_features'])

# Menghitung Cosine Similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Menentukan film yang akan digunakan sebagai referensi
movie_title = 'King Kong'
idx = df[df['title'] == movie_title].index

# Mendapatkan skor kemiripan untuk film yang ditonton
sim_scores = list(enumerate(cosine_sim[idx]))

# Mengurutkan film berdasarkan skor kemiripan tertinggi
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

# Mendapatkan indeks film yang paling mirip
movie_indices = [i[0] for i in sim_scores[1:11]]  # Mengambil 10 film teratas

# Mendapatkan rekomendasi film
recommendations = df.iloc[movie_indices][['title', 'genres', 'overview']]
print(recommendations)

Empty DataFrame
Columns: [title, genres, overview]
Index: []


In [77]:
mlb = MultiLabelBinarizer()
genres_binarized = mlb.fit_transform(df['genres'])
genres_df = pd.DataFrame(genres_binarized, columns=mlb.classes_, index=df['id'])
item_feature_matrix = genres_df.copy()

item_feature_matrix['vote_average'] = np.random.uniform(5, 10, size=len(df))

print("Item Feature Matrix:\n", item_feature_matrix)

user_item_data = {
    'user': ['user 1', 'user 2', 'user 3', 'user 4'],
    951: [9, 4, 6, 7],
    136: [9, 7, 9, 7],
    235: [7, 8, 9, 7],
    918: [5, 9, 5, 7]
}

df_user_items = pd.DataFrame(user_item_data)
df_user_items.set_index('user', inplace=True)

print("\nUser-Item Rating Matrix:\n", df_user_items)

rated_item_ids = df_user_items.columns
df_rated_items = df[df['id'].isin(rated_item_ids)]
item_feature_matrix_rated = item_feature_matrix.loc[rated_item_ids]

print("\nItem Feature Matrix for Rated Items:\n", item_feature_matrix_rated)

user_feature_matrix = df_user_items.T.dot(item_feature_matrix_rated)
user_feature_matrix = user_feature_matrix.div(df_user_items.T.notna().sum(), axis=1)

print("\nUser Feature Matrix:\n", user_feature_matrix)

user_feature_vector = user_feature_matrix.mean(axis=1)

predicted_scores = item_feature_matrix.dot(user_feature_vector)

df_items_with_scores = df.set_index('id').join(predicted_scores.rename('predicted_score'))

top_recommendations = {}
for user in df_user_items.index:
    user_ratings = df_user_items.loc[user]
    unseen_items = df_items_with_scores[~df_items_with_scores.index.isin(user_ratings.index)]
    top_recommendations[user] = unseen_items.sort_values(by='predicted_score', ascending=False).head(10)

print("\nTop 10 Recommendations for Each User:\n", top_recommendations)

Item Feature Matrix:
            A  C  D  F  H  M  R  S  T  V  W  a  c  d  e  g  h  i  l  m  n  o  \
id                                                                            
19995   1  1  0  0  1  0  0  0  1  0  0  0  1  1  1  1  0  0  1  0  0  1  1   
285     1  1  0  0  1  0  0  0  0  0  0  0  1  1  1  1  0  0  1  0  0  1  1   
206647  1  1  1  0  0  0  0  0  0  0  0  0  0  1  1  1  0  0  1  0  1  1  1   
49026   1  1  1  1  0  0  0  0  0  1  0  0  1  1  0  1  0  1  1  1  1  1  1   
49529   1  1  0  0  1  0  0  0  1  0  0  0  0  1  1  1  0  0  1  0  0  1  1   
...    .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. .. ..   
9367    1  1  1  0  0  0  0  0  0  1  0  0  0  1  0  1  0  1  1  1  1  1  1   
72766   1  0  1  0  0  0  0  1  0  0  0  0  1  1  1  1  0  0  0  0  1  1  1   
231617  1  0  1  1  0  0  1  1  0  1  1  0  1  1  1  1  0  0  1  0  1  1  1   
126186  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0   
25975   0  0  0  1  0  0  0  0

KeyError: '[136, 918] not in index'